# unicomplex 00 · `collective status`

Six stores are kept fresh by scheduled jobs (the Massive, SEC and FINRA syncs, the vinculum
build), a risk model rebuilt by hand and an execution journal. `collective status` reads each
store's own status through its public API and compares it with what the schedule should have
produced by now. Nothing here downloads, builds, connects to a broker or writes.

1. the report
2. the expectations behind each check
3. one failing store
4. machine-readable output

In [1]:
import json
from datetime import UTC, datetime

import polars as pl
from subspace_probe import style, timed
from unicomplex import STATES, collect, render

style()
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(90)
with timed("collect()"):
    checks = collect()
print(render(checks))

collect(): 1.16 s
collective status at 2026-11-04 17:05 ET
source             check           state  last        expected                      detail
massive            bars_day        ok     2026-11-03  2026-11-03                    5825 days stored
massive            bars_minute     ok     2026-11-03  2026-11-03                    5825 days stored
massive            stream          ok     2026-11-04  running                       pid 56538, connected=True, last error: ConnectionClosedError: no close frame received or sent
edgar              facts           ok     2026-11-03  2026-11-01 or later           20,469 companies, bulk 2026-11-01, last sync 2026-11-04T12:19
edgar              series          ok     -           built from the current facts  materialized series current
finra              short_interest  ok     2026-10-15  2026-10-09 or later           212 settlement dates
finra              short_volume    ok     2026-11-03  2026-11-03                    4341 days
vinculum     

## 1 · The report

Each line is a `Check`: the source package, the check, its state, the latest date or time the
store holds, what it should hold by now, and a detail. The states:

In [2]:
print(STATES)
pl.DataFrame([c.to_dict() for c in checks])

('ok', 'stale', 'info', 'missing')


source,name,state,last,expected,detail
str,str,str,str,str,str
"""massive""","""bars_day""","""ok""","""2026-11-03""","""2026-11-03""","""5825 days stored"""
"""massive""","""bars_minute""","""ok""","""2026-11-03""","""2026-11-03""","""5825 days stored"""
"""massive""","""stream""","""ok""","""2026-11-04""","""running""","""pid 56538, connected=True, last error: ConnectionClosedError: no close frame received or s…"
"""edgar""","""facts""","""ok""","""2026-11-03""","""2026-11-01 or later""","""20,469 companies, bulk 2026-11-01, last sync 2026-11-04T12:19"""
"""edgar""","""series""","""ok""",null,"""built from the current facts""","""materialized series current"""
"""finra""","""short_interest""","""ok""","""2026-10-15""","""2026-10-09 or later""","""212 settlement dates"""
"""finra""","""short_volume""","""ok""","""2026-11-03""","""2026-11-03""","""4341 days"""
"""vinculum""","""market_cap""","""ok""","""2026-11-03""","""2026-11-03""","""built 2026-11-04T12:48, 31,544,348 rows"""
"""borg-queen""","""risk_model""","""ok""","""2026-11-02""","""2026-09-20 or later""","""built 2026-11-03T19:50, 3,962 days of factor returns"""


## 2 · The expectations behind each check

| Check | Fresh when |
|---|---|
| massive `bars_day`, `bars_minute` | the last stored day is the last completed NYSE session before today (the 06:30 sync) |
| massive `stream` | the stream service is running and, in market hours, has received a message in the last 30 minutes |
| edgar `facts` | `sync_through` is within 3 days of today (the sync runs Tue-Sat through yesterday) |
| edgar `series` | the materialized series were built from the current facts |
| finra `short_interest` | the last settlement date is within 26 days (two cycles plus the 7-business-day publication lag) |
| finra `short_volume` | the last day is the last completed session |
| vinculum `market_cap` | the panel runs through the last completed session (`vinculum build` after the syncs) |
| borg-queen `risk_model` | the last month end in the model is within 45 days (it is rebuilt by hand) |
| transwarp-conduit `orders` | informational: the last order sheet and the fills journaled |

The thresholds follow the schedules documented in each package; a schedule change there means a
threshold change in `unicomplex`.

## 3 · One failing store

`collect` takes store objects, so a notebook (or a test) can swap one out. A vinculum that is not
built yet becomes one `missing` line with its error, and the other checks still run:

In [3]:
class NotBuilt:
    def status(self):
        raise FileNotFoundError("market cap table is not built yet. Run: vinculum build")


demo = collect(vinculum=NotBuilt(), now=datetime.now(UTC))
print(render([c for c in demo if c.source in ("vinculum", "edgar")]))

collective status at 2026-11-04 17:05 ET
source    check       state    last        expected                      detail
edgar     facts       ok       2026-11-03  2026-11-01 or later           20,469 companies, bulk 2026-11-01, last sync 2026-11-04T12:19
edgar     series      ok       -           built from the current facts  materialized series current
vinculum  market_cap  missing  -           -                             FileNotFoundError: market cap table is not built yet. Run: vinculum build
1 check(s) need attention: vinculum/market_cap


## 4 · Machine-readable output

`collective status --json` prints the same checks as JSON, and the command exits 1 when anything
is stale or missing, so `tools/nightly_sync.sh`, cron or a shell prompt can act on it.

In [4]:
print(json.dumps([c.to_dict() for c in checks[:2]], indent=1))
print("exit code would be", 1 if any(c.state in ("stale", "missing") for c in checks) else 0)

[
 {
  "source": "massive",
  "name": "bars_day",
  "state": "ok",
  "last": "2026-11-03",
  "expected": "2026-11-03",
  "detail": "5825 days stored"
 },
 {
  "source": "massive",
  "name": "bars_minute",
  "state": "ok",
  "last": "2026-11-03",
  "expected": "2026-11-03",
  "detail": "5825 days stored"
 }
]
exit code would be 0


## Takeaways
- One command answers "is everything a rebalance depends on as fresh as it should be?".
- Expectations are schedules, not ages: on a Monday morning the expected session is Friday's.
- Phase 2 of this package is scheduled runs with one log and one lock, replacing the
  per-package jobs; the status command is its first building block.